# Observações:
### Este código deverá rodar somente com os dados de um único usuário por vez, modificando a variavel `user_id` de acordo com o dado inserido previamente no banco de dados.

In [3]:
import json
import psycopg2
import os

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    "host": DB_HOST,
    "dbname": DB_NAME,
    "user": DB_USER,
    "password": DB_PASSWORD,
    "sslmode": "require"
}

def format_date(date_str):
    """Ajusta datas como '2026-01' para '2026-01-01' para o PostgreSQL"""
    if len(date_str) == 7:  # Formato YYYY-MM
        return f"{date_str}-01"
    return date_str

def sync_sequences(cursor):
    """Sincroniza os contadores de ID para evitar erros de 'duplicate key'"""
    tables_to_sync = [
        ('"User"', 'user_id'),
        ('"SoundCapsule"', 'id')
    ]
    for table, column in tables_to_sync:
        cursor.execute(f"""
            SELECT setval(pg_get_serial_sequence('{table}', '{column}'), 
            coalesce(max({column}), 0) + 1, false) FROM {table};
        """)

def insert_data():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()

        with open("YourSoundCapsule.json", "r", encoding="utf-8") as f:
            data = json.load(f)

        # 1. Ajusta os ponteiros de ID
        sync_sequences(cursor)

        # 2. Usuário
        user_id = 3

        # 3. Processar as Stats
        for entry in data.get("stats", []):
            # CORREÇÃO DA DATA AQUI
            entry_date = format_date(entry["date"])
            
            cursor.execute('SELECT id FROM "SoundCapsule" WHERE user_id = %s AND "date" = %s;', (user_id, entry_date))
            sc_row = cursor.fetchone()
            
            if sc_row:
                sc_id = sc_row[0]
            else:
                cursor.execute("""
                    INSERT INTO "SoundCapsule" ("user_id", "date", "streamCount", "secondsPlayed")
                    VALUES (%s, %s, %s, %s) RETURNING id;
                """, (user_id, entry_date, entry.get("streamCount", 0), entry.get("secondsPlayed", 0)))
                sc_id = cursor.fetchone()[0]

            print("SoundCapsuleId Adquirido")

            # 4. Top Tracks
            for track in entry.get("topTracks", []):
                t_name = str(track["name"]).replace("'", "''")

                cursor.execute(f"SELECT id FROM \"Tracks\" WHERE track_name = '{t_name}' LIMIT 1")
                t_id = cursor.fetchone()
                if not t_id:
                    cursor.execute(f"INSERT INTO \"Tracks\" (track_name) VALUES ('{t_name}') ON CONFLICT (id) DO NOTHING RETURNING id;")
                    t_id = cursor.fetchone()

                t_id = t_id[0]

                cursor.execute('SELECT 1 FROM "SoundCapsule_TopTracks" WHERE soundcapsule_id = %s AND track_id = %s;', (sc_id, t_id))
                if not cursor.fetchone():
                    cursor.execute('INSERT INTO "SoundCapsule_TopTracks" (soundcapsule_id, track_id, "streamCount") VALUES (%s, %s, %s);', 
                                   (sc_id, t_id, track.get("streamCount", 0)))
            
            print("TopTracks feito")

            # 5. Top Artists
            for artist in entry.get("topArtists", []):
                a_name = str(artist["name"]).replace("'", "''")
                cursor.execute(f"SELECT id FROM \"Artist\" WHERE artist_name = '{a_name}' LIMIT 1")
                a_id = cursor.fetchone()
                if not a_id:
                    cursor.execute(f"INSERT INTO \"Artist\" (artist_name) VALUES ('{a_name}') ON CONFLICT (id) DO NOTHING RETURNING id;")
                    a_id = cursor.fetchone()

                a_id = a_id[0]

                cursor.execute('SELECT 1 FROM "SoundCapsule_TopArtists" WHERE soundcapsule_id = %s AND artist_id = %s;', (sc_id, a_id))
                if not cursor.fetchone():
                    cursor.execute('INSERT INTO "SoundCapsule_TopArtists" (soundcapsule_id, artist_id, "streamCount") VALUES (%s, %s, %s);', 
                                   (sc_id, a_id, artist.get("streamCount", 0)))
                    
            print("TopArtists feito")

            # 6. Top Genres
            for genre in entry.get("topGenres", []):
                g_name = genre["name"]
                cursor.execute('SELECT 1 FROM "SoundCapsule_TopGenres" WHERE soundcapsule_id = %s AND genre_name = %s;', (sc_id, g_name))
                if not cursor.fetchone():
                    cursor.execute('INSERT INTO "SoundCapsule_TopGenres" (soundcapsule_id, genre_name) VALUES (%s, %s);', (sc_id, g_name))

            print("TopGenres feito")

            # 7. Time of Day
            for tod in entry.get("timeOfDayStats", []):
                cursor.execute('SELECT 1 FROM "SoundCapsule_TimeOfDayStats" WHERE soundcapsule_id = %s AND period = %s;', (sc_id, tod["period"]))
                if not cursor.fetchone():
                    cursor.execute('INSERT INTO "SoundCapsule_TimeOfDayStats" (soundcapsule_id, period, "secondsPlayed") VALUES (%s, %s, %s);', 
                                   (sc_id, tod["period"], tod["secondsPlayed"]))
                    
            print("Time of Day feito")

        conn.commit()
        print("Dados inseridos com sucesso! 🚀")

    except Exception as e:
        if 'conn' in locals(): conn.rollback()
        print(f"Erro ao inserir dados: {e}")
    finally:
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()

if __name__ == "__main__":
    insert_data()

SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time of Day feito
SoundCapsuleId Adquirido
TopTracks feito
TopArtists feito
TopGenres feito
Time o